<a href="https://colab.research.google.com/github/apopuri584/wriggling_bad2/blob/main/caffeine_multithreaded_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
import os, csv, glob, sys, random, time, threading, queue, re, shutil, subprocess
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import cv2, torch, numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_fill_holes
from skimage.morphology import skeletonize
from skimage.measure import regionprops
%matplotlib inline

try:
    from segment_anything import sam_model_registry, SamPredictor
except ImportError:
    !pip install git+https://github.com/facebookresearch/segment-anything.git -q
    from segment_anything import sam_model_registry, SamPredictor

if not os.path.exists('sam_vit_b_01ec64.pth'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print('SAM checkpoint downloaded.')
else:
    print('SAM checkpoint already present.')

from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))


  Preparing metadata (setup.py) ... done
SAM checkpoint downloaded.


In [5]:


from google.colab import drive
drive.mount('/content/drive')


DRIVE_INPUT_FOLDER = '/content/drive/MyDrive/C. elegans videos/Caffeine treatment (M9 buffer +0.01% Tween 20)'
OUTPUT_ROOT         = '/content/drive/MyDrive/C. elegans videos/Segmented_Output_Caffeine'

# Only the 10 mM / 20 mM caffeine subfolders are processed. Folder names on
# disk vary (e.g. '10mM caffeine', '10 mM', '20mM', '20  MM Caffeine'), so
# every subfolder name is scanned for a "<number> mM" pattern anywhere in the
# name and normalized to a canonical '10 mM' / '20 mM' label -- that
# canonical label is what's used both to select input folders and to name
# the matching output subfolder.
def normalize_conc_name(name):
    m = re.search(r'(\d+)\s*m\s*m', name, flags=re.IGNORECASE)
    return f'{m.group(1)} mM' if m else None

_all_subfolders = sorted([
    d for d in glob.glob(os.path.join(DRIVE_INPUT_FOLDER, '*'))
    if os.path.isdir(d)
])

CONDITION_FOLDERS = []   # list of (input_folder_path, canonical_condition_name)
for d in _all_subfolders:
    canon = normalize_conc_name(os.path.basename(d))
    if canon in ('10 mM', '20 mM'):
        CONDITION_FOLDERS.append((d, canon))

if not CONDITION_FOLDERS:
    print(f'⚠ No 10 mM / 20 mM subfolders found under: {DRIVE_INPUT_FOLDER}')
    print('  Double-check DRIVE_INPUT_FOLDER above and that the Drive shortcut was added.')
else:
    print('Found condition folders:')
    for d, canon in CONDITION_FOLDERS:
        print(f' - {os.path.basename(d)}  ->  {canon}')

VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv', '.wmv')

def list_videos(folder):
    return sorted([
        f for f in glob.glob(os.path.join(folder, '*'))
        if f.lower().endswith(VIDEO_EXTENSIONS)
    ])


N_BG_FRAMES        = 30
BG_THRESH          = None   # None = Otsu auto-threshold
MIN_WORM_AREA      = 50
MAX_WORM_AREA      = 50_000
MIN_ASPECT_RATIO   = 1.2
MIN_SOLIDITY       = 0.15
BOX_PADDING        = 10
TRACKING_IOU_THRESH = 0.1

SKELETON_WORKERS = min(8, (os.cpu_count() or 4))
FRAME_PREFETCH   = 8

# ── Resume / skip behaviour ────────────────────────────────────────────────
# An existing segmented output shorter than this is assumed to have been cut
# off mid-run (e.g. a Colab disconnect) and is resumed/completed rather than
# skipped or restarted from scratch.
MIN_VALID_DURATION_SEC = 5.0

VIDEO_WORKERS = min(4, os.cpu_count() or 2)

# ── Accuracy guards against exploding / spreading masks ────────────────────
# During fast worm movement (or when two worms cross paths) a single bad SAM
# mask can suddenly balloon onto a neighbour or a background artifact, which
# then wrecks the skeleton for the rest of the track. These thresholds bound
# how much a track's mask is allowed to change frame-to-frame; anything
# outside the bounds is rejected for that frame (see enforce_mask_consistency
# in the helper-functions cell).
MAX_AREA_GROWTH_RATIO = 1.8   # reject a mask more than this x the track's
                              # rolling median area
MAX_AREA_SHRINK_RATIO = 0.4   # reject a mask below this fraction of the
                              # track's rolling median area
AREA_HISTORY_LEN      = 8     # frames of area history kept per track
MAX_CENTROID_JUMP_PX  = 60    # reject a match whose centroid jumps further
                              # than this in a single frame (px)

WORM_COLORS = [
    (0,   255, 0),    # green
    (255, 100, 0),    # orange
    (0,   180, 255),  # cyan
    (255, 0,   200),  # magenta
    (255, 255, 0),    # yellow
    (150, 0,   255),  # purple
    (0,   255, 180),  # teal
    (255, 50,  50),   # red
    (50,  50,  255),  # blue
    (255, 200, 0),    # gold
]

os.makedirs(OUTPUT_ROOT, exist_ok=True)
for _, canon in CONDITION_FOLDERS:
    os.makedirs(os.path.join(OUTPUT_ROOT, canon), exist_ok=True)
print(f'\nOutput root : {OUTPUT_ROOT}')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ── Cell 3: Load SAM model ────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('⚠ WARNING: No GPU detected! SAM will be extremely slow on CPU.')
    print('  In Colab: Runtime → Change runtime type → Hardware accelerator → GPU (T4), then re-run all cells.')

sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b_01ec64.pth')
sam.to(device)
predictor = SamPredictor(sam)
print('SAM model loaded.')

# The predictor holds mutable state (the current image embedding) between
# .set_image() and .predict_torch(), so it cannot safely be called from two
# threads at once. Every video-processing thread acquires this lock around
# its SAM calls; everything else about a video's processing stays concurrent.
SAM_LOCK = threading.Lock()
print('✓ SAM_LOCK created for thread-safe SAM inference.')


In [ ]:
# ── Cell 4: Helper functions (original + resume, concat, and accuracy guards) ─

def build_background(video_path, n_frames=N_BG_FRAMES):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // n_frames)
    frames = []
    for i in range(0, min(total, n_frames * step), step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    if not frames:
        raise RuntimeError('Could not read any frames for background estimation.')
    return np.median(np.stack(frames, axis=0), axis=0).astype(np.uint8)


def detect_worm_boxes(gray_frame, background,
                      bg_thresh=BG_THRESH,
                      min_area=MIN_WORM_AREA, max_area=MAX_WORM_AREA,
                      min_aspect=MIN_ASPECT_RATIO, min_solidity=MIN_SOLIDITY,
                      padding=BOX_PADDING):
    h, w = gray_frame.shape
    bright = cv2.subtract(gray_frame, background)
    if bg_thresh is None:
        _, fg_mask = cv2.threshold(bright, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, fg_mask = cv2.threshold(bright, bg_thresh, 255, cv2.THRESH_BINARY)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    num_labels, label_img, stats, _ = cv2.connectedComponentsWithStats(fg_mask, connectivity=8)
    boxes = []
    for lbl in range(1, num_labels):
        area = stats[lbl, cv2.CC_STAT_AREA]
        if not (min_area <= area <= max_area):
            continue
        blob  = (label_img == lbl).astype(np.uint8)
        props = regionprops(blob)[0]
        aspect = (props.major_axis_length / props.minor_axis_length
                  if props.minor_axis_length > 0 else 0)
        if aspect < min_aspect or props.solidity < min_solidity:
            continue
        x  = stats[lbl, cv2.CC_STAT_LEFT]
        y  = stats[lbl, cv2.CC_STAT_TOP]
        bw = stats[lbl, cv2.CC_STAT_WIDTH]
        bh = stats[lbl, cv2.CC_STAT_HEIGHT]
        boxes.append(np.array([max(0, x-padding), max(0, y-padding),
                                 min(w-1, x+bw+padding), min(h-1, y+bh+padding)]))
    return boxes, fg_mask

def segment_boxes_batched(predictor, image_rgb, boxes):

    predictor.set_image(image_rgb)
    if len(boxes) == 0:
        return []
    boxes_np = np.stack(boxes).astype(np.float32)
    boxes_t  = torch.as_tensor(boxes_np, device=predictor.device)
    transformed_boxes = predictor.transform.apply_boxes_torch(boxes_t, image_rgb.shape[:2])
    with torch.no_grad():
        masks, _, _ = predictor.predict_torch(
            point_coords=None,
            point_labels=None,
            boxes=transformed_boxes,
            multimask_output=False,
        )
    masks = masks.squeeze(1).to(torch.uint8).cpu().numpy()  # (N, H, W)
    return [masks[i] for i in range(masks.shape[0])]


def clean_mask_for_tracking(mask):
    """Reduce a raw SAM mask to its single largest connected component.

    SAM occasionally returns a mask with a small disconnected speck (a bit of
    background clutter, a neighbouring worm's edge, etc). Left in, that speck
    can pull the skeleton off the worm's body or add a spurious branch/
    endpoint. Keeping only the largest blob keeps the skeleton anchored to
    the actual worm.
    """
    mask_u8 = (mask > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num <= 2:  # background + at most one foreground component
        return mask_u8
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_lbl = 1 + int(np.argmax(areas))
    return (labels == largest_lbl).astype(np.uint8)


def enforce_mask_consistency(new_tracks, prev_tracks, area_hist, last_centroid):
    """Guard against a mask exploding, collapsing, or teleporting in one frame.

    For every track, the incoming mask is cleaned (largest component only)
    and checked against that track's own recent history:
      - area must stay within [MAX_AREA_SHRINK_RATIO, MAX_AREA_GROWTH_RATIO]
        times the rolling median area, and
      - the centroid must not jump more than MAX_CENTROID_JUMP_PX in one frame.
    A track that fails either check is frozen to its last trustworthy mask
    for this frame, instead of letting a spurious blob spread through (and
    corrupt) the rest of that worm's track.
    """
    cleaned = {}
    for tid, mask in new_tracks.items():
        mask = clean_mask_for_tracking(mask)
        ys, xs = np.where(mask == 1)

        if len(xs) == 0:
            if tid in prev_tracks:
                cleaned[tid] = prev_tracks[tid]
            continue

        area = int(mask.sum())
        cx, cy = float(xs.mean()), float(ys.mean())

        ok = True
        hist = area_hist.get(tid)
        if hist:
            med = float(np.median(hist))
            if med > 0 and not (MAX_AREA_SHRINK_RATIO * med <= area <= MAX_AREA_GROWTH_RATIO * med):
                ok = False
        if tid in last_centroid:
            pcx, pcy = last_centroid[tid]
            if np.hypot(cx - pcx, cy - pcy) > MAX_CENTROID_JUMP_PX:
                ok = False

        if ok:
            cleaned[tid] = mask
            area_hist.setdefault(tid, deque(maxlen=AREA_HISTORY_LEN)).append(area)
            last_centroid[tid] = (cx, cy)
        elif tid in prev_tracks:
            cleaned[tid] = prev_tracks[tid]   # freeze -- reject this frame's mask
        else:
            # Brand-new track with nothing to compare against yet -- accept
            # and start building its history.
            cleaned[tid] = mask
            area_hist.setdefault(tid, deque(maxlen=AREA_HISTORY_LEN)).append(area)
            last_centroid[tid] = (cx, cy)
    return cleaned, area_hist, last_centroid


def skeletonize_mask(mask):
    ys, xs = np.where(mask == 1)
    skeleton = np.zeros(mask.shape, dtype=bool)
    if len(xs) == 0:
        return skeleton, 0, [], 0.0

    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    crop = mask[y0:y1, x0:x1].astype(bool)
    skel_crop = skeletonize(crop)
    skeleton[y0:y1, x0:x1] = skel_crop

    length_px = int(skel_crop.sum())
    endpoints = []
    skel_pts_crop = np.argwhere(skel_crop)
    if len(skel_pts_crop) > 2:
        for (r, c) in skel_pts_crop:
            patch = skel_crop[max(0,r-1):r+2, max(0,c-1):c+2]
            if patch.sum() == 2:
                endpoints.append((r + y0, c + x0))

    curvature = 0.0
    if len(skel_pts_crop) >= 3:
        pca_vec = np.cov(skel_pts_crop.T)
        _, evecs = np.linalg.eigh(pca_vec)
        proj    = skel_pts_crop @ evecs[:, -1]
        ordered = skel_pts_crop[np.argsort(proj)]
        sampled = ordered[::5]
        if len(sampled) >= 3:
            angles = []
            for i in range(1, len(sampled) - 1):
                v1 = sampled[i] - sampled[i-1]
                v2 = sampled[i+1] - sampled[i]
                cos_a = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2) + 1e-8)
                angles.append(np.arccos(np.clip(cos_a, -1, 1)))
            curvature = float(np.mean(angles)) if angles else 0.0
    return skeleton, length_px, endpoints, curvature


def mask_iou(m1, m2):
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / union if union > 0 else 0.0


def match_masks_to_tracks(prev_masks_dict, new_masks,
                           iou_thresh=TRACKING_IOU_THRESH,
                           next_id_ref=[0]):
    updated, used_new = {}, set()
    scores = [(mask_iou(pm, nm), tid, j)
               for tid, pm in prev_masks_dict.items()
               for j, nm in enumerate(new_masks)]
    scores.sort(reverse=True)
    matched_tids = set()
    for iou, tid, j in scores:
        if iou < iou_thresh: break
        if tid in matched_tids or j in used_new: continue
        updated[tid] = new_masks[j]; matched_tids.add(tid); used_new.add(j)
    for j, nm in enumerate(new_masks):
        if j not in used_new:
            updated[next_id_ref[0]] = nm; next_id_ref[0] += 1
    return updated


def contour_skeleton_and_overlap(sam_mask, sam_skeleton):
    ys, xs = np.where(sam_mask == 1)
    contour_skel = np.zeros(sam_mask.shape, dtype=bool)
    if len(xs) == 0:
        return contour_skel, 0, 0.0, 0.0

    pad = 2  # room for the dilate/erode below
    y0 = max(0, ys.min() - pad); y1 = min(sam_mask.shape[0], ys.max() + 1 + pad)
    x0 = max(0, xs.min() - pad); x1 = min(sam_mask.shape[1], xs.max() + 1 + pad)

    mask_crop = sam_mask[y0:y1, x0:x1]
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    dilated = cv2.dilate(mask_crop, kernel, iterations=1)
    eroded  = cv2.erode(mask_crop,  kernel, iterations=1)
    filled  = binary_fill_holes(cv2.subtract(dilated, eroded).astype(bool)).astype(np.uint8)
    contour_skel_crop = skeletonize(filled.astype(bool))
    contour_skel[y0:y1, x0:x1] = contour_skel_crop

    A, B = sam_skeleton.astype(bool), contour_skel.astype(bool)
    intersection = int(np.logical_and(A, B).sum())
    a_sum, b_sum = int(A.sum()), int(B.sum())
    dice    = (2*intersection)/(a_sum+b_sum) if (a_sum+b_sum) > 0 else 0.0
    iou_val = intersection/int(np.logical_or(A,B).sum()) if np.logical_or(A,B).any() else 0.0
    return contour_skel, intersection, dice, iou_val


def overlay_masks(frame_bgr, track_dict, track_skeletons=None, alpha=0.5):
    overlay = frame_bgr.copy()
    for tid, mask in track_dict.items():
        color_bgr = WORM_COLORS[tid % len(WORM_COLORS)][::-1]
        overlay[mask == 1] = color_bgr
        ys, xs = np.where(mask == 1)
        if len(xs) == 0: continue
        cx, cy = int(xs.mean()), int(ys.mean())
        if track_skeletons and tid in track_skeletons:
            for sy, sx in zip(*np.where(track_skeletons[tid])):
                cv2.circle(overlay, (int(sx), int(sy)), 1, (255,255,255), -1)
        cv2.putText(overlay, f'W{tid}', (cx-10, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)
    # Fixed: Added the missing 'gamma' argument (usually 0 for simple blending)
    return cv2.addWeighted(frame_bgr, 1-alpha, overlay, alpha, 0)



def _compute_track_metrics(tid_mask):
    tid, mask = tid_mask
    skel, length_px, endpoints, curvature = skeletonize_mask(mask)
    contour_skel, overlap_px, dice, iou_val = contour_skeleton_and_overlap(mask, skel)
    ys, xs = np.where(mask == 1)
    if len(xs) == 0:
        return None
    cx, cy = float(xs.mean()), float(ys.mean())
    return {
        'tid': tid, 'skel': skel,
        'length_px': length_px, 'endpoints': endpoints, 'curvature': curvature,
        'contour_skel_px': int(contour_skel.sum()), 'overlap_px': overlap_px,
        'dice': dice, 'iou_val': iou_val, 'cx': cx, 'cy': cy,
    }


def compute_all_track_metrics(tracks, executor):
    """Fan the per-worm skeletonization/scoring for this frame out across
    the thread pool and gather results (worms with an empty mask are dropped,
    same as the original per-worm 'continue' behaviour)."""
    results = list(executor.map(_compute_track_metrics, tracks.items()))
    return [r for r in results if r is not None]




class FrameReaderThread(threading.Thread):
    def __init__(self, video_path, start_frame=0, queue_size=FRAME_PREFETCH):
        super().__init__(daemon=True)
        self.cap = cv2.VideoCapture(video_path)
        if start_frame:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        self.q = queue.Queue(maxsize=queue_size)
        self._stop_flag = False

    def run(self):
        while not self._stop_flag:
            ret, frame = self.cap.read()
            if not ret:
                self.q.put(None)
                break
            self.q.put(frame)

    def get_frame(self):
        return self.q.get()

    def stop(self):
        self._stop_flag = True
        self.cap.release()


# ── Resume-support helpers ──────────────────────────────────────────────────

def get_video_duration(path):
    """Return (frame_count, fps, duration_sec) for an existing video file.
    Returns (0, 0.0, 0.0) if the file can't be opened/read."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return 0, 0.0, 0.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.release()
    duration = frame_count / fps if fps > 0 else 0.0
    return frame_count, fps, duration


def _last_csv_frame(csv_path):
    """Return the highest 'frame' value logged in an existing motion_data.csv,
    or -1 if the file is missing/empty/unreadable."""
    if not os.path.exists(csv_path):
        return -1
    last = -1
    try:
        with open(csv_path, 'r', newline='') as f:
            reader = csv.reader(f)
            next(reader, None)  # header
            for row in reader:
                if row:
                    last = max(last, int(row[0]))
    except Exception:
        return -1
    return last


def concat_videos_stream_copy(part_paths, output_path):
    """Concatenate videos with identical codec/size/fps via ffmpeg's concat
    demuxer using stream copy (fast, no re-encode)."""
    list_file = output_path + '.concat_list.txt'
    with open(list_file, 'w') as f:
        for p in part_paths:
            f.write(f"file '{os.path.abspath(p)}'\n")
    cmd = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', list_file, '-c', 'copy', output_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    try:
        os.remove(list_file)
    except OSError:
        pass
    if result.returncode != 0:
        raise RuntimeError(result.stderr.decode(errors='ignore')[-1000:])


def concat_csv_files(backup_csv_path, continuation_csv_path, final_csv_path):
    """Concatenate a completed CSV (with header) and a continuation CSV
    (whose own header is skipped) into final_csv_path."""
    with open(final_csv_path, 'w', newline='') as out_f:
        with open(backup_csv_path, 'r', newline='') as f1:
            shutil.copyfileobj(f1, out_f)
        with open(continuation_csv_path, 'r', newline='') as f2:
            next(f2, None)  # skip header
            shutil.copyfileobj(f2, out_f)


print('✓ All helper functions defined (multithreading + resume + accuracy guards).')


In [ ]:
# ── Cell 5: Core per-video processing function ─────────────────────────────

def process_video(video_path, out_dir):
    """
    Run the full SAM segmentation + tracking pipeline on a single video.
    Outputs are saved to out_dir:
      - <name>.segmented_multi.avi   (annotated video)
      - <name>.motion_data.csv       (per-frame worm measurements)

    Resume / skip:
      - If a finished output already exists (duration >= MIN_VALID_DURATION_SEC),
        this video is skipped entirely.
      - If an existing output is SHORTER than MIN_VALID_DURATION_SEC (the sign
        of a run that was cut off, e.g. by a Colab disconnect), processing
        resumes right after the last frame actually written, and the new
        frames are stitched onto the existing partial output via ffmpeg
        stream-copy rather than reprocessing the video from scratch.

    Concurrency:
      - This function is safe to call from multiple threads at once (see the
        batch cell). The SAM predictor is stateful and NOT thread-safe, so
        every call into it is serialized with SAM_LOCK; video decoding,
        background modeling, box detection, skeletonization, and disk I/O for
        different videos still run fully concurrently.
      - Within a single video: a FrameReaderThread prefetches/decodes frames
        while the GPU works on the previous frame, and a ThreadPoolExecutor
        (SKELETON_WORKERS threads) skeletonizes + scores every worm in a
        frame in parallel.

    Accuracy guards:
      - Every mask is reduced to its largest connected component
        (clean_mask_for_tracking) before it's used, so a stray SAM blob can't
        drag the skeleton off the worm's body.
      - enforce_mask_consistency rejects any single-frame mask that balloons,
        collapses, or has a centroid that jumps implausibly far relative to
        that track's own recent history, and freezes the track to its last
        trustworthy mask instead -- this is what stops the segmentation from
        "exploding" or spreading onto a neighbour/background during fast
        worm movement.

    Returns one of: 'ok', 'skipped', 'no_worms', 'error'.
    """
    os.makedirs(out_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    out_video_path = os.path.join(out_dir, base_name + '.segmented_multi.avi')
    out_csv_path   = os.path.join(out_dir, base_name + '.motion_data.csv')

    resume_frame      = 0
    backup_video_path = None
    backup_csv_path   = None

    # ── Resume / skip decision ─────────────────────────────────────────────
    if os.path.exists(out_video_path):
        existing_frames, _, existing_dur = get_video_duration(out_video_path)

        if existing_dur >= MIN_VALID_DURATION_SEC:
            print(f'  ⏭  Skipping (already complete, {existing_dur:.1f}s): {base_name}')
            return 'skipped'

        csv_last_frame = _last_csv_frame(out_csv_path)
        candidate_resume = csv_last_frame + 1 if csv_last_frame >= 0 else existing_frames
        resume_frame = max(0, min(candidate_resume, existing_frames))

        if resume_frame > 0:
            print(f'  ↻  Incomplete output found ({existing_dur:.1f}s < '
                  f'{MIN_VALID_DURATION_SEC:.0f}s) -- resuming at frame {resume_frame}: {base_name}')
            backup_video_path = out_video_path + '.partial_backup.avi'
            backup_csv_path   = out_csv_path + '.partial_backup.csv'
            if os.path.exists(backup_video_path):
                os.remove(backup_video_path)
            if os.path.exists(backup_csv_path):
                os.remove(backup_csv_path)
            shutil.move(out_video_path, backup_video_path)
            if os.path.exists(out_csv_path):
                shutil.move(out_csv_path, backup_csv_path)
        else:
            print(f'  ↻  Incomplete output found but unusable -- restarting from scratch: {base_name}')
            os.remove(out_video_path)
            if os.path.exists(out_csv_path):
                os.remove(out_csv_path)

    # Open video & read metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  ERROR: cannot open {video_path}')
        return 'error'
    fps          = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    resume_note = f' | resuming at frame {resume_frame}' if resume_frame else ''
    print(f'  Video: {width}x{height} @ {fps} fps | {total_frames} frames{resume_note}')

    # Build background model (always from the full original video)
    print('  Building background model …')
    background = build_background(video_path)

    # Detect & segment the init frame (frame 0, or the resume frame)
    cap = cv2.VideoCapture(video_path)
    if resume_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, resume_frame)
    ret, first_bgr = cap.read()
    cap.release()
    if not ret:
        print('  ERROR: cannot read init frame.')
        return 'error'

    first_rgb  = cv2.cvtColor(first_bgr, cv2.COLOR_BGR2RGB)
    first_gray = cv2.cvtColor(first_bgr, cv2.COLOR_BGR2GRAY)
    init_boxes, _ = detect_worm_boxes(first_gray, background)

    if len(init_boxes) == 0:
        print('  WARNING: no worms detected in init frame — skipping this video.')
        print('           Try adjusting MIN_ASPECT_RATIO / MIN_SOLIDITY / BG_THRESH.')
        return 'no_worms'

    if len(init_boxes) > 100:
        print(f'  WARNING: {len(init_boxes)} candidate worms detected. Skipping.')
        return 'no_worms'

    print(f'  Detected {len(init_boxes)} worm(s) in init frame.')

    with SAM_LOCK:
        init_masks = segment_boxes_batched(predictor, first_rgb, init_boxes)
    init_masks = [clean_mask_for_tracking(m) for m in init_masks]

    next_id = [len(init_masks)]
    tracks  = {i: m for i, m in enumerate(init_masks)}

    # Continuation output goes to its own temp path so the backed-up partial
    # run is never touched until the two are safely stitched together.
    work_video_path = out_video_path + '.continued.avi' if backup_video_path else out_video_path
    work_csv_path   = out_csv_path   + '.continued.csv' if backup_csv_path   else out_csv_path

    executor = ThreadPoolExecutor(max_workers=SKELETON_WORKERS)
    try:
        track_skeletons     = {}
        track_area_hist     = {}
        track_last_centroid = {}
        for r in compute_all_track_metrics(tracks, executor):
            track_skeletons[r['tid']] = r['skel']
            track_area_hist[r['tid']] = deque([int(tracks[r['tid']].sum())], maxlen=AREA_HISTORY_LEN)
            track_last_centroid[r['tid']] = (r['cx'], r['cy'])

        print(f'{len(tracks)} tracks and {len(track_skeletons)} track skeletons in init frame.')

        # ── Set up output writer & CSV ────────────────────────────────────
        fourcc     = cv2.VideoWriter_fourcc(*'XVID')
        out_writer = cv2.VideoWriter(work_video_path, fourcc, fps, (width, height))
        csv_file   = open(work_csv_path, 'w', newline='')
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow([
            'frame', 'worm_id',
            'centroid_x', 'centroid_y',
            'speed_px_per_frame', 'heading_deg',
            'body_length_px', 'curvature_rad_per_step',
            'n_endpoints',
            'contour_skel_px', 'skel_overlap_px', 'skel_dice', 'skel_iou',
        ])

        # Write the (already-processed) init frame
        blended = overlay_masks(first_bgr, tracks, track_skeletons)
        out_writer.write(blended)

        # also write a still image of the init frame - in case writer doesn't complete
        out_frame0_result_path = os.path.join(out_dir, base_name + '.frame0_result.png')
        cv2.imwrite(out_frame0_result_path, blended)

        reader = FrameReaderThread(video_path, start_frame=resume_frame + 1)
        reader.start()

        prev_centroids = dict(track_last_centroid)
        frame_idx = resume_frame + 1
        t_start = time.time()

        try:
            while True:
                frame_bgr = reader.get_frame()
                if frame_bgr is None:
                    break

                frame_rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                frame_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

                # Bounding boxes from existing tracks
                track_boxes, track_box_ids = [], []
                for tid, mask in tracks.items():
                    ys, xs = np.where(mask == 1)
                    if len(xs) == 0: continue
                    x1 = max(0, xs.min()-BOX_PADDING); y1 = max(0, ys.min()-BOX_PADDING)
                    x2 = min(width-1, xs.max()+BOX_PADDING); y2 = min(height-1, ys.max()+BOX_PADDING)
                    track_boxes.append(np.array([x1, y1, x2, y2]))
                    track_box_ids.append(tid)

                # Connected-component detection
                det_boxes, _ = detect_worm_boxes(frame_gray, background)

                # Merge boxes (deduplicate with tracked)
                all_boxes = list(track_boxes)
                for db in det_boxes:
                    overlap = any(
                        (lambda ix1, iy1, ix2, iy2: (ix2>ix1 and iy2>iy1 and
                         (ix2-ix1)*(iy2-iy1)/((db[2]-db[0])*(db[3]-db[1])+1e-6) > 0.5))(
                             max(db[0],tb[0]), max(db[1],tb[1]),
                             min(db[2],tb[2]), min(db[3],tb[3]))
                        for tb in track_boxes
                    )
                    if not overlap:
                        all_boxes.append(db)

                if not all_boxes:
                    out_writer.write(frame_bgr)
                    frame_idx += 1
                    continue

                # One batched SAM call for all boxes in this frame (GPU-serialized)
                with SAM_LOCK:
                    new_masks = segment_boxes_batched(predictor, frame_rgb, all_boxes)

                # Match to tracks
                prev_tracks_snapshot = tracks
                candidate_tracks = match_masks_to_tracks(tracks, new_masks,
                                                iou_thresh=TRACKING_IOU_THRESH,
                                                next_id_ref=next_id)

                # ACCURACY GUARD: clean masks + reject any that explode,
                # collapse, or teleport relative to that track's own history.
                tracks, track_area_hist, track_last_centroid = enforce_mask_consistency(
                    candidate_tracks, prev_tracks_snapshot, track_area_hist, track_last_centroid)

                # Skeletonize + score every worm this frame in parallel.
                track_skeletons = {}
                for r in compute_all_track_metrics(tracks, executor):
                    tid = r['tid']
                    track_skeletons[tid] = r['skel']
                    cx, cy = r['cx'], r['cy']
                    if tid in prev_centroids:
                        pcx, pcy = prev_centroids[tid]
                        speed   = float(np.sqrt((cx-pcx)**2 + (cy-pcy)**2))
                        heading = float(np.degrees(np.arctan2(cy-pcy, cx-pcx)))
                    else:
                        speed, heading = 0.0, 0.0
                    prev_centroids[tid] = (cx, cy)
                    csv_writer.writerow([
                        frame_idx, tid,
                        round(cx, 2), round(cy, 2),
                        round(speed, 3), round(heading, 2),
                        r['length_px'], round(r['curvature'], 4),
                        len(r['endpoints']),
                        r['contour_skel_px'], r['overlap_px'],
                        round(r['dice'], 4), round(r['iou_val'], 4),
                    ])

                # Write annotated frame
                out_writer.write(overlay_masks(frame_bgr, tracks, track_skeletons))
                frame_idx += 1

                if frame_idx % 25 == 0 or frame_idx == total_frames:
                    elapsed = time.time() - t_start
                    rate = (frame_idx - resume_frame) / elapsed if elapsed > 0 else 0
                    eta_min = (total_frames - frame_idx) / rate / 60 if rate > 0 else float('nan')
                    print(f'  Frame {frame_idx}/{total_frames} | active tracks: {len(tracks)} '
                          f'| {rate:.1f} fps | ETA {eta_min:.1f} min', flush=True)
        finally:
            reader.stop()
    finally:
        executor.shutdown(wait=True)

    out_writer.release()
    csv_file.close()

    # ── Stitch the resumed segment back onto the backed-up partial output ──
    if backup_video_path:
        try:
            concat_videos_stream_copy([backup_video_path, work_video_path], out_video_path)
        except Exception as e:
            print(f'  ⚠ Could not merge with previous partial video ({e}). '
                  f'Keeping the new segment separately as {os.path.basename(work_video_path)}.')
        else:
            os.remove(backup_video_path)
            os.remove(work_video_path)

        try:
            concat_csv_files(backup_csv_path, work_csv_path, out_csv_path)
        except Exception as e:
            print(f'  ⚠ Could not merge with previous partial CSV ({e}). '
                  f'Keeping the new segment separately as {os.path.basename(work_csv_path)}.')
        else:
            os.remove(backup_csv_path)
            os.remove(work_csv_path)

    print(f'  ✓ Saved: {os.path.basename(out_video_path)}')
    print(f'  ✓ Saved: {os.path.basename(out_csv_path)}')
    return 'ok'


print('✓ process_video() function defined (multithreaded, resumable, accuracy-guarded).')


In [ ]:
# ── Cell 6: Run the batch pipeline (multithreaded across all videos) ───────
# Every video in both the 10 mM and 20 mM Caffeine treatment folders is queued
# up and processed by a pool of VIDEO_WORKERS threads. Real GPU work (SAM
# inference) is serialized via SAM_LOCK so results stay correct, but frame
# decoding, background modeling, box detection, skeletonization, and disk I/O
# for different videos overlap -- so multiple videos make progress at once
# instead of one at a time.

PRINT_LOCK = threading.Lock()

def safe_print(*args, **kwargs):
    with PRINT_LOCK:
        print(*args, **kwargs)

video_jobs = []   # (condition_name, video_path, out_dir)
for condition_folder, canon_name in CONDITION_FOLDERS:
    out_dir = os.path.join(OUTPUT_ROOT, canon_name)
    for video_path in list_videos(condition_folder):
        video_jobs.append((canon_name, video_path, out_dir))

print('═' * 60)
print(f'Found {len(video_jobs)} video(s) across {len(CONDITION_FOLDERS)} condition folder(s).')
print(f'Processing with {VIDEO_WORKERS} worker thread(s) (SAM inference itself is serialized).')
print('═' * 60)

def _run_job(job):
    canon_name, video_path, out_dir = job
    safe_print(f'\n[{canon_name}] Processing: {os.path.basename(video_path)}')
    safe_print('─' * 60)
    try:
        status = process_video(video_path, out_dir)
        if status == 'ok':
            safe_print(f'✓ Done: {os.path.basename(video_path)}')
        elif status == 'skipped':
            pass  # process_video already logged the skip
        elif status == 'no_worms':
            safe_print(f'⚠ No worms detected / video skipped: {os.path.basename(video_path)}')
    except Exception as e:
        safe_print(f'✗ ERROR on {os.path.basename(video_path)}: {e}')
        status = 'error'
    return status

results = {'ok': 0, 'skipped': 0, 'no_worms': 0, 'error': 0}
if video_jobs:
    with ThreadPoolExecutor(max_workers=VIDEO_WORKERS) as batch_executor:
        for status in batch_executor.map(_run_job, video_jobs):
            results[status] = results.get(status, 0) + 1

print(f'\n\nBatch complete:')
print(f"  ✓ Processed : {results.get('ok', 0)}")
print(f"  ⏭ Skipped   : {results.get('skipped', 0)}  (already complete)")
print(f"  ⚠ No worms  : {results.get('no_worms', 0)}")
print(f"  ✗ Errors    : {results.get('error', 0)}")
print(f"  Total       : {len(video_jobs)}")


In [ ]:
# ── Cell 7: Verify output files ─────────────────────────────────────────────

print(f'Output folder: {OUTPUT_ROOT}\n')
for root, dirs, files_ in os.walk(OUTPUT_ROOT):
    dirs.sort(); files_.sort()
    level = root.replace(OUTPUT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files_:
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f'{indent}  {f}  ({size_mb:.1f} MB)')


In [ ]:
# ── Cell 8 (Optional): Quick summary plots for all CSVs ───────────────────────
# Loads every motion_data.csv and plots per-condition speed distributions.

import pandas as pd
from scipy.stats import gaussian_kde

csv_files = []
for root, _, files in os.walk(OUTPUT_ROOT):
    for f in files:
        if f.endswith('.motion_data.csv'):
            condition = os.path.basename(root)
            csv_files.append((condition, os.path.join(root, f)))

if not csv_files:
    print('No CSV files found yet — run the batch pipeline first.')
else:
    all_data = []
    for condition, csv_path in csv_files:
        df = pd.read_csv(csv_path)
        df['condition'] = condition
        df['source_file'] = os.path.basename(csv_path)
        all_data.append(df)
    combined = pd.concat(all_data, ignore_index=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    conditions = combined['condition'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(conditions)))

    for ax, metric, label in zip(
        axes,
        ['speed_px_per_frame', 'body_length_px', 'curvature_rad_per_step'],
        ['Speed (px/frame)', 'Body length (px)', 'Curvature (rad/step)']
    ):
        for cond, color in zip(conditions, colors):
            sub = combined[combined['condition'] == cond][metric].dropna()
            if not sub.empty:
                kde = gaussian_kde(sub)
                x_vals = np.linspace(sub.min(), sub.max(), 200)
                ax.plot(x_vals, kde(x_vals), label=cond, color=color)
                ax.fill_between(x_vals, kde(x_vals), alpha=0.2, color=color)
        ax.set_xlabel(label)
        ax.set_ylabel('Density')
        ax.legend()

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_ROOT, 'summary_plots.png')
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'✓ Saved: {plot_path}')


In [ ]:
# ── Cell 9 (Optional): Download a local zip of all results ─────────────────
# Everything is already saved directly to your Google Drive folder above, so
# this is only needed if you also want a local copy on this machine.

import shutil
from google.colab import files as colab_files

zip_path = shutil.make_archive('/content/segmented_output', 'zip', OUTPUT_ROOT)
colab_files.download(zip_path)
